# Evaluate explanations: Efficiency + Simplicity

First phase of the metrics evaluation (per the agreed build order: Efficiency + Simplicity -> Fidelity -> Robustness -> Stability). Both metrics here are computed entirely from **already-saved** explanation artifacts -- no new SHAP/LIME calls.

**Normalization for cross-dataset comparability** (the 6 datasets range from 6 to 84+ features):
- **Efficiency**: `runtime_ms_per_instance_per_feature` -- normalized by *model-input* dimensionality (post one-hot encoding), since that's the actual computational cost driver.
- **Simplicity**: `normalized_entropy_complexity` (bounded [0,1] via `/log2(n_features)`) and `pct_features_for_80/90pct_mass` -- both normalized by *original* feature count (one-hot dummies rolled back up via the feature-group map), since a 3-category variable shouldn't count as 3x as complex as a numeric one.

See `src/evaluation.py`'s module docstring for the full reasoning, including why `num_features` deliberately means different things for the two metrics.

Output: `Evaluation/metrics_long.csv` (long-format, append-only -- see note at the bottom) and `Evaluation/run_ledger.csv`.

In [1]:
import sys
sys.path.insert(0, "..")
from src.evaluation import create_experiment, run_efficiency_and_simplicity
from src.utils import Logger


In [2]:
import os
log_path = '../Evaluation/logs/ledger.log'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(log_path), exist_ok=True)

In [3]:
DATASETS = [
    ("healthcare", "pima_diabetes"),
    ("healthcare", "breast_cancer_wisconsin"),
    ("healthcare", "heart_disease_uci"),
    ("finance", "loan_default"),
    ("finance", "financial_distress"),
    ("finance", "credit_card_fraud_2023"),
]
MODEL_NAMES = ["rf", "xgb"]

logger = Logger("../Evaluation/logs", filename="ledger.log")
experiment_id = create_experiment(
    "../Evaluation",
    {"phase": "efficiency_simplicity", "datasets": DATASETS, "models": MODEL_NAMES},
    logger,
)
print("experiment_id:", experiment_id)


Created experiment_id=exp_20260712T205452_d673a94a, logged to ../Evaluation\run_ledger.csv
experiment_id: exp_20260712T205452_d673a94a


In [4]:
all_rows = []
for domain, dataset_name in DATASETS:
    for model_name in MODEL_NAMES:
        print(f"\n{'='*70}\nEVALUATING: {dataset_name} x {model_name}\n{'='*70}")
        config = {
            "dataset_name": dataset_name,
            "domain": domain,
            "model_name": model_name,
            "experiment_id": experiment_id,
            "datasets_root": "../Datasets",
            "explanations_root": "../Explanations",
            "evaluation_root": "../Evaluation",
        }
        all_rows += run_efficiency_and_simplicity(config)

print(f"\nTotal rows appended this run: {len(all_rows)}")



EVALUATING: pima_diabetes x rf

------------------------------------------------------------
EFFICIENCY + SIMPLICITY: pima_diabetes x rf
------------------------------------------------------------
Feature-group map for pima_diabetes: 8 original features (from 8 model-input columns).
Efficiency (shap, pima_diabetes, rf): 0.260 ms/instance, 0.03247 ms/instance/feature
SHAP simplicity: 154 instances x 3 metrics = 462 rows.
Efficiency (lime, pima_diabetes, rf): 27.597 ms/instance, 3.44968 ms/instance/feature
LIME simplicity: 154 instances x 3 metrics = 462 rows (0 unparsed conditions excluded).
Appended 928 row(s) to ../Evaluation\metrics_long.csv

------------------------------------------------------------
DONE
------------------------------------------------------------

EVALUATING: pima_diabetes x xgb

------------------------------------------------------------
EFFICIENCY + SIMPLICITY: pima_diabetes x xgb
------------------------------------------------------------
Feature-group map

In [5]:
import pandas as pd
df = pd.read_csv("../Evaluation/metrics_long.csv")
df[df.experiment_id == experiment_id].groupby(["dataset", "model", "explainer", "metric_name"])["value"].mean().unstack("metric_name")


metric_name                              normalized_entropy_complexity  \
dataset                 model explainer                                  
breast_cancer_wisconsin rf    lime                            0.878999   
                              shap                            0.822059   
                        xgb   lime                            0.881595   
                              shap                            0.850673   
credit_card_fraud_2023  rf    lime                            0.833094   
                              shap                            0.735816   
                        xgb   lime                            0.867711   
                              shap                            0.815322   
financial_distress      rf    lime                            0.831642   
                              shap                            0.798485   
                        xgb   lime                            0.833984   
                              shap                            0.819140   
heart_disease_uci       rf    lime                            0.838506   
                              shap                            0.799345   
                        xgb   lime                            0.901200   
                              shap                            0.851638   
loan_default            rf    lime                            0.859376   
                              shap                            0.854822   
                        xgb   lime                            0.974559   
                              shap                            0.912014   
pima_diabetes           rf    lime                            0.785129   
                              shap                            0.736528   
                        xgb   lime                            0.762220   
                              shap                            0.709658   

metric_name                              pct_features_for_80pct_mass  \
dataset                 model explainer                                
breast_cancer_wisconsin rf    lime                         45.467836   
                              shap                         36.286550   
                        xgb   lime                         44.853801   
                              shap                         39.678363   
credit_card_fraud_2023  rf    lime                         39.365517   
                              shap                         26.772414   
                        xgb   lime                         45.979310   
                              shap                         38.827586   
financial_distress      rf    lime                         34.334047   
                              shap                         25.095692   
                        xgb   lime                         36.304658   
                              shap                         31.523291   
heart_disease_uci       rf    lime                         49.498328   
                              shap                         43.060201   
                        xgb   lime                         56.396321   
                              shap                         49.790970   
loan_default            rf    lime                         46.300000   
                              shap                         48.087500   
                        xgb   lime                         74.462500   
                              shap                         56.650000   
pima_diabetes           rf    lime                         50.081169   
                              shap                         44.237013   
                        xgb   lime                         48.782468   
                              shap                         42.451299   

metric_name                              pct_features_for_90pct_mass  \
dataset                 model explainer                                
breast_cancer_wisconsin rf    lime                        

### Note: metrics_long.csv is append-only
Re-running this notebook creates a **new** `experiment_id` and appends another full set of rows rather than overwriting -- this is deliberate (per the logging design, re-derivable results with revised definitions later is a real scenario), but it means duplicate-looking data will accumulate across reruns. Filter by `experiment_id` (see `run_ledger.csv` for what each one was) to select which run's numbers to analyze. Let us know if you'd rather this step be idempotent (skip datasets/models already covered) instead.